In [16]:
import numpy as np
import itertools
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta, timezone
from datetime import datetime, timedelta, timezone
import requests



In [5]:
end_date = datetime(2026, 3, 1, tzinfo=timezone.utc)
print(end_date)
start_date = end_date - timedelta(days=365*3)
print(start_date)
start_str = start_date.strftime('%Y-%m-%d')
end_str = end_date.strftime('%Y-%m-%d')

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

2026-03-01 00:00:00+00:00
2023-03-02 00:00:00+00:00


In [8]:

# Your 8x8 grid (64 points)
lats = np.linspace(49.0, 60.0, 11) 
lons = np.linspace(-9.0, 2.0, 11)  
grid_points = list(itertools.product(lats, lons))
api_lats = [p[0] for p in grid_points]
api_lons = [p[1] for p in grid_points]


params = {
    "latitude": api_lats,    
    "longitude": api_lons,   
    "start_date": start_str, 
    "end_date": end_str,
    "hourly": ["temperature_2m", "wind_speed_100m", "cloud_cover", "wind_direction_100m"]
}

responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)

print("Parsing and averaging grid weather data...")
all_grid_data = []

for response in responses:
    hourly = response.Hourly()
    # Create a dataframe for this specific grid coordinate
    df_coord = pd.DataFrame({
        "time": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temp": hourly.Variables(0).ValuesAsNumpy(),
        "wind_speed": hourly.Variables(1).ValuesAsNumpy(),
        "clouds": hourly.Variables(2).ValuesAsNumpy(),
        "wind_dir": hourly.Variables(3).ValuesAsNumpy()
    })
    all_grid_data.append(df_coord)


weather_df = pd.concat(all_grid_data).groupby("time").mean().reset_index()

# win dir encoding
wind_dir_rad = weather_df['wind_dir'] * np.pi / 180
weather_df['wind_sin'] = np.sin(wind_dir_rad)
weather_df['wind_cos'] = np.cos(wind_dir_rad)

weather_df = weather_df.drop(columns=['wind_dir'])



Parsing and averaging grid weather data...


In [9]:
all_grid_data

[                           time   temp  wind_speed  clouds    wind_dir
 0     2023-03-02 00:00:00+00:00   9.00   19.130875   100.0   70.201042
 1     2023-03-02 01:00:00+00:00   9.05   20.140705   100.0   65.725571
 2     2023-03-02 02:00:00+00:00   8.50   19.513195    63.0   60.124096
 3     2023-03-02 03:00:00+00:00   8.55   15.683774    48.0   58.134064
 4     2023-03-02 04:00:00+00:00   8.60   15.546833    94.0   47.815628
 ...                         ...    ...         ...     ...         ...
 26299 2026-03-01 19:00:00+00:00  12.10   77.890717   100.0  190.115295
 26300 2026-03-01 20:00:00+00:00  12.15   79.247444   100.0  189.676208
 26301 2026-03-01 21:00:00+00:00  12.20   78.863007   100.0  191.053421
 26302 2026-03-01 22:00:00+00:00  10.95   39.377647   100.0  298.087799
 26303 2026-03-01 23:00:00+00:00  10.30   29.516706   100.0  291.087769
 
 [26304 rows x 5 columns],
                            time   temp  wind_speed  clouds    wind_dir
 0     2023-03-02 00:00:00+00:00   

In [10]:
all_locations_data = []

for i, response in enumerate(responses):
    lat = round(api_lats[i])
    lon = round(api_lons[i])
    lon_str = str(lon).replace('-', 'm') # minus = m
    lat_str = str(lat).replace('-', 'm')
    
    loc_suffix = f"lat{lat_str}_lon_{lon_str}"

    hourly = response.Hourly()
    
    time = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )
    raw_wind_dir_deg = hourly.Variables(3).ValuesAsNumpy()
    
    #  degrees to radians 
    wind_dir_rad = raw_wind_dir_deg * (np.pi / 180)
    data = {
        "time": time,
        f"temp_{loc_suffix}": hourly.Variables(0).ValuesAsNumpy(),
        f"wind_{loc_suffix}": hourly.Variables(1).ValuesAsNumpy(),
        f"clouds_{loc_suffix}": hourly.Variables(2).ValuesAsNumpy(),

        f"wind_sin_{loc_suffix}": np.sin(wind_dir_rad),
        f"wind_cos_{loc_suffix}": np.cos(wind_dir_rad)
    }
    
    all_locations_data.append(pd.DataFrame(data).set_index("time"))

final_df = pd.concat(all_locations_data, axis=1)
final_df = final_df.reset_index(names="time") 

print(f"Shape {final_df.shape}")


Shape (26304, 606)


In [11]:
final_df.columns

Index(['time', 'temp_lat49_lon_m9', 'wind_lat49_lon_m9', 'clouds_lat49_lon_m9',
       'wind_sin_lat49_lon_m9', 'wind_cos_lat49_lon_m9', 'temp_lat49_lon_m8',
       'wind_lat49_lon_m8', 'clouds_lat49_lon_m8', 'wind_sin_lat49_lon_m8',
       ...
       'temp_lat60_lon_1', 'wind_lat60_lon_1', 'clouds_lat60_lon_1',
       'wind_sin_lat60_lon_1', 'wind_cos_lat60_lon_1', 'temp_lat60_lon_2',
       'wind_lat60_lon_2', 'clouds_lat60_lon_2', 'wind_sin_lat60_lon_2',
       'wind_cos_lat60_lon_2'],
      dtype='object', length=606)

In [65]:
final_df.to_csv('grid_weather_11x11_2023-2026.csv', index=False)

In [12]:


def get_elexon_dataset(dataset_name, start_time, end_time):
    """Fetches a specific time window from Elexon."""
    url = f"https://data.elexon.co.uk/bmrs/api/v1/datasets/{dataset_name}"
    params = {"from": start_time, "to": end_time, "format": "json"}
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if 'data' in data and data['data']:
            return pd.DataFrame(data['data'])
    else:
        print(f"Error {response.status_code}: {response.text}")
    return pd.DataFrame() # Return empty DF on failure

def fetch_month_of_data(dataset_name,start_date,end_date):
    """Loops through the last 30 days in 7-day chunks."""
 
    
    current_start = start_date
    all_chunks = []
    
    
    while current_start < end_date:

        current_end = min(current_start + timedelta(days=7), end_date)
        
        # F Elexon's requirement: YYYY-MM-DDTHH:MMZ
        start_str = current_start.strftime('%Y-%m-%dT%H:%MZ')
        end_str = current_end.strftime('%Y-%m-%dT%H:%MZ')
        
        print(f"Fetching chunk: {start_str} to {end_str}")
        chunk_df = get_elexon_dataset(dataset_name, start_str, end_str)
        print(chunk_df)
        if not chunk_df.empty:
            all_chunks.append(chunk_df)
            
        current_start = current_end
        
    # Combine all the weekly chunks into one DataFrame
    if all_chunks:
        master_df = pd.concat(all_chunks, ignore_index=True)
        
        # Cort chronologically
        if 'startTime' in master_df.columns:
            master_df['startTime'] = pd.to_datetime(master_df['startTime'])
            master_df = master_df.sort_values('startTime').reset_index(drop=True)
            
        return master_df
    else:
        print("No data was fetched.")






In [ ]:
#  1 month of Market Index Data (MID)
monthly_mid_df = fetch_month_of_data("MID", start_date, end_date)
monthly_mid_df=monthly_mid_df.loc[monthly_mid_df['dataProvider']=='APXMIDP']


In [14]:
def fetch_availability_month(start_date, end_date):
    """Fetches FOU2T14D using the correct publishDateTime parameters."""
    current_start = start_date
    all_chunks = []
    
    
    while current_start < end_date:
        current_end = min(current_start + timedelta(days=7), end_date)
        
        start_str = current_start.strftime('%Y-%m-%dT%H:%MZ')
        end_str = current_end.strftime('%Y-%m-%dT%H:%MZ')
        
        print(f"Fetching chunk: {start_str} to {end_str}")
        
        url = "https://data.elexon.co.uk/bmrs/api/v1/datasets/FOU2T14D"
        params = {
            "publishDateTimeFrom": start_str,
            "publishDateTimeTo": end_str,
            "format": "json"
        }
        
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            if 'data' in data and data['data']:
                all_chunks.append(pd.DataFrame(data['data']))
        else:
            print(f"Error {response.status_code}: {response.text}")
            
        current_start = current_end
        
    if all_chunks:
        return pd.concat(all_chunks, ignore_index=True)
    return pd.DataFrame()




In [17]:

avail_raw = fetch_availability_month(start_date, end_date)


Fetching chunk: 2023-03-02T00:00Z to 2023-03-09T00:00Z
Fetching chunk: 2023-03-09T00:00Z to 2023-03-16T00:00Z
Fetching chunk: 2023-03-16T00:00Z to 2023-03-23T00:00Z
Fetching chunk: 2023-03-23T00:00Z to 2023-03-30T00:00Z
Fetching chunk: 2023-03-30T00:00Z to 2023-04-06T00:00Z
Fetching chunk: 2023-04-06T00:00Z to 2023-04-13T00:00Z
Fetching chunk: 2023-04-13T00:00Z to 2023-04-20T00:00Z
Fetching chunk: 2023-04-20T00:00Z to 2023-04-27T00:00Z
Fetching chunk: 2023-04-27T00:00Z to 2023-05-04T00:00Z
Fetching chunk: 2023-05-04T00:00Z to 2023-05-11T00:00Z
Fetching chunk: 2023-05-11T00:00Z to 2023-05-18T00:00Z
Fetching chunk: 2023-05-18T00:00Z to 2023-05-25T00:00Z
Fetching chunk: 2023-05-25T00:00Z to 2023-06-01T00:00Z
Fetching chunk: 2023-06-01T00:00Z to 2023-06-08T00:00Z
Fetching chunk: 2023-06-08T00:00Z to 2023-06-15T00:00Z
Fetching chunk: 2023-06-15T00:00Z to 2023-06-22T00:00Z
Fetching chunk: 2023-06-22T00:00Z to 2023-06-29T00:00Z
Fetching chunk: 2023-06-29T00:00Z to 2023-07-06T00:00Z
Fetching c

In [18]:

if not avail_raw.empty:
    
    avail_raw['publishTime'] = pd.to_datetime(avail_raw['publishTime'], utc=True)
    avail_raw['forecastDate'] = pd.to_datetime(avail_raw['forecastDate'], utc=True)
    
    delta = avail_raw['forecastDate'].dt.normalize() - avail_raw['publishTime'].dt.normalize()
    avail_raw['days_ahead'] = delta.dt.days
    
    avail_day_ahead = avail_raw[avail_raw['days_ahead'] == 2].copy()
    
    avail_clean = avail_day_ahead.groupby('forecastDate')['outputUsable'].sum().reset_index()
    avail_clean = avail_clean.rename(columns={'forecastDate': 'time', 'outputUsable': 'available_mw'})
    avail_clean.set_index('time', inplace=True)
    

else:
    print("Failed to fetch FOU2T14D.")
avail_clean

,available_mw
time,
2023-03-04 00:00:00+00:00,1212485
2023-03-05 00:00:00+00:00,1231482
2023-03-06 00:00:00+00:00,1408762
2023-03-07 00:00:00+00:00,1318548
2023-03-08 00:00:00+00:00,1271853
...,...
2026-02-27 00:00:00+00:00,1449945
2026-02-28 00:00:00+00:00,1426063
2026-03-01 00:00:00+00:00,1598385


In [19]:
import yfinance as yf
def get_commodities(days=30):
    """ Gas and Oil prices from Yahoo Finance."""
    gas = yf.Ticker("TTF=F").history(period=f"{days}d") # gas
    oil = yf.Ticker("BZ=F").history(period=f"{days}d") # Brent Crude
    return gas, oil

def fetch_commodities_month(start_str, end_str):
    print(f"Fetching Commodity Prices: {start_str} to {end_str}...")
    
    # TTF=F European Gas Benchmark, BZ=F  Brent Crude
    tickers = ["TTF=F", "BZ=F"]
    
    # yfinance's end date is exclusive
    yf_end = (end_date + timedelta(days=1)).strftime('%Y-%m-%d')
    
    df = yf.download(tickers, start=start_str, end=yf_end, progress=False)
    
    if 'Close' in df.columns:
        close_df = df['Close'].reset_index()
    # else:
    #     # Fallback for some yfinance version differences
    #     close_df = df.xs('Close', axis=1, level=0).reset_index() if isinstance(df.columns, pd.MultiIndex) else df.reset_index()
        
    close_df = close_df.rename(columns={"TTF=F": "gas_price", "BZ=F": "oil_price", "Date": "time"})
    
    # Timezone UTC datetime
    close_df['time'] = pd.to_datetime(close_df['time']).dt.tz_localize('UTC')
    return close_df
commodities_df = fetch_commodities_month(start_str, end_str)

Fetching Commodity Prices: 2023-03-02 to 2026-03-01...


C:\Users\steph\AppData\Local\Temp\ipykernel_32584\1158000229.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, start=start_str, end=yf_end, progress=False)


In [17]:
commodities_df

Ticker,time,oil_price,gas_price
0,2022-03-02 00:00:00+00:00,112.930000,165.542999
1,2022-03-03 00:00:00+00:00,110.459999,160.822998
2,2022-03-04 00:00:00+00:00,118.110001,192.550003
3,2022-03-07 00:00:00+00:00,123.209999,227.201004
4,2022-03-08 00:00:00+00:00,127.980003,214.554001
...,...,...,...
1000,2026-02-23 00:00:00+00:00,71.489998,31.834000
1001,2026-02-24 00:00:00+00:00,70.769997,30.891001
1002,2026-02-25 00:00:00+00:00,70.849998,31.049999
1003,2026-02-26 00:00:00+00:00,70.750000,32.223999


In [18]:
monthly_mid_df

,dataset,startTime,dataProvider,settlementDate,settlementPeriod,price,volume
1,MID,2022-03-02 00:00:00+00:00,APXMIDP,2022-03-02,1,194.59,859.90
2,MID,2022-03-02 00:30:00+00:00,APXMIDP,2022-03-02,2,196.44,1208.55
4,MID,2022-03-02 01:00:00+00:00,APXMIDP,2022-03-02,3,171.47,949.40
6,MID,2022-03-02 01:30:00+00:00,APXMIDP,2022-03-02,4,182.31,1747.60
8,MID,2022-03-02 02:00:00+00:00,APXMIDP,2022-03-02,5,178.03,1073.55
...,...,...,...,...,...,...,...
140234,MID,2026-02-28 22:00:00+00:00,APXMIDP,2026-02-28,45,90.83,2331.40
140236,MID,2026-02-28 22:30:00+00:00,APXMIDP,2026-02-28,46,88.31,2552.95
140239,MID,2026-02-28 23:00:00+00:00,APXMIDP,2026-02-28,47,89.46,2607.10
140241,MID,2026-02-28 23:30:00+00:00,APXMIDP,2026-02-28,48,89.33,2137.65


In [19]:
monthly_mid_df.groupby('startTime').agg({'price': 'mean', 'volume': 'sum'}).reset_index()

,startTime,price,volume
0,2022-03-02 00:00:00+00:00,194.59,859.90
1,2022-03-02 00:30:00+00:00,196.44,1208.55
2,2022-03-02 01:00:00+00:00,171.47,949.40
3,2022-03-02 01:30:00+00:00,182.31,1747.60
4,2022-03-02 02:00:00+00:00,178.03,1073.55
...,...,...,...
70063,2026-02-28 22:00:00+00:00,90.83,2331.40
70064,2026-02-28 22:30:00+00:00,88.31,2552.95
70065,2026-02-28 23:00:00+00:00,89.46,2607.10
70066,2026-02-28 23:30:00+00:00,89.33,2137.65


In [ ]:

mid_clean = monthly_mid_df.rename(columns={'startTime': 'time'})
mid_clean['time'] = pd.to_datetime(mid_clean['time'], utc=True)
mid_clean.set_index('time', inplace=True)

mid_hourly = mid_clean.resample('1H').agg({'price': 'mean', 'volume': 'sum'})

C:\Users\steph\AppData\Local\Temp\ipykernel_33808\2872879525.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  mid_hourly = mid_clean.resample('1H').agg({'price': 'mean', 'volume': 'sum'})


In [ ]:
# avail_clean = availability_df.copy()
# avail_clean['time'] = pd.to_datetime(avail_clean['forecastDate'], utc=True)
# # If there are multiple zones reporting for the same day, sum them up
# avail_clean = avail_clean.groupby('time')['outputUsable'].sum().reset_index()
# avail_clean.set_index('time', inplace=True)

In [ ]:
comm_clean = commodities_df.copy()
comm_clean['time'] = pd.to_datetime(comm_clean['time'], utc=True)
comm_clean.set_index('time', inplace=True)

In [40]:
master_df = mid_hourly.join(comm_clean, how='left') \
                      .join(avail_clean, how='left')

In [44]:
cols_to_ffill = ['oil_price', 'gas_price', 'available_mw']
master_df[cols_to_ffill] = master_df[cols_to_ffill].ffill()

In [54]:
master_df['volume'] = master_df['volume'].replace(0.0, np.nan)

In [48]:
master_df.isna().sum()

price            2
volume           0
oil_price        0
gas_price        0
available_mw    48
dtype: int64

In [56]:
master_df[['price', 'volume']] = master_df[['price', 'volume']].interpolate(method='linear')

In [59]:
master_df.dropna(inplace=True)

In [61]:
master_df.to_csv('price_commo_avail_power_2023_2026.csv',index=True)